# DBSCAN & Pipelines — Hands-on Tutorial

K-means needed you to pick K and assumed round clusters. **DBSCAN** drops both
assumptions: it finds clusters as **dense regions** separated by sparse gaps, decides
the number of clusters itself, and labels sparse points as **noise**.

In this notebook:

1. DBSCAN on shapes K-means can't handle
2. **Core / border / noise** — the three kinds of point
3. Tuning **`eps`** and **`min_samples`**
4. The **k-distance plot** — a systematic way to pick `eps`
5. **Failure modes** — variable density and chaining
6. A **pipeline**: PCA first, *then* cluster

This follows the *DBSCAN & Pipelines* lecture (03_b).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

np.random.seed(42)

# A colour map that paints noise (label -1) grey
CMAP = {-1: '#bdc3c7', 0: '#2980b9', 1: '#e74c3c', 2: '#27ae60',
        3: '#f39c12', 4: '#8e44ad'}
def dbcolors(labels):
    return [CMAP.get(l, '#000000') for l in labels]

---
## Part 1: DBSCAN on two moons + noise

Two interleaving crescents with scattered noise — K-means would slice straight
through them. DBSCAN follows the density instead.

In [ ]:
X, _ = make_moons(n_samples=200, noise=0.07, random_state=42)
noise = np.random.uniform([-1.5, -0.8], [2.5, 1.3], size=(15, 2))
X_all = np.vstack([X, noise])

plt.scatter(X_all[:, 0], X_all[:, 1], s=20, color='#7f8c8d', alpha=0.7)
plt.title('Raw data — where are the clusters?')
plt.xticks([]); plt.yticks([])
plt.show()

### Exercise: run DBSCAN

Fit `DBSCAN(eps=0.2, min_samples=5)` on `X_all`, then report how many clusters it
found and how many points it called noise.

**Hints:**
- `db = DBSCAN(eps=0.2, min_samples=5).fit(X_all)`; labels are `db.labels_`.
- Noise points have label `-1`.
- Number of clusters = number of distinct non-negative labels:
  `len(set(labels)) - (1 if -1 in labels else 0)`.

In [ ]:
# YOUR CODE HERE
# Fit DBSCAN, then compute n_clusters and n_noise.
raise NotImplementedError("Fit DBSCAN and summarise the result")

plt.scatter(X_all[:, 0], X_all[:, 1], s=20, c=dbcolors(labels), alpha=0.7)
plt.title(f'{n_clusters} clusters, {n_noise} noise points')
plt.xticks([]); plt.yticks([])
plt.show()

### Think about it

- DBSCAN found the two crescents without being told there were two. Where did the
  scattered noise points end up?
- Notice you never passed a number of clusters. What you *did* pass was `eps` and
  `min_samples` — those define what "dense enough" means. Part 3.

---
## Part 2: Core, border, and noise points

DBSCAN sorts every point into one of three kinds, using `eps` (a radius) and
`min_samples` (a count):

- **Core**: has at least `min_samples` points within `eps` — it sits in a dense region.
- **Border**: not core itself, but within `eps` of a core point — the edge of a cluster.
- **Noise**: neither — too isolated to belong anywhere.

Clusters grow by linking core points whose `eps`-neighbourhoods overlap.

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider
from matplotlib.lines import Line2D

# A small hand-placed set: a dense clump (cores + borders), one edge point,
# and two isolated outliers. Small enough that every neighbourhood is legible.
P = np.array([
    [0.00, 0.00], [0.30, 0.05], [-0.25, 0.20], [0.10, -0.25],
    [0.35, 0.30], [-0.10, -0.10],   # the clump
    [0.70, 0.50],                   # edge point
    [1.60, 1.20], [-1.30, -0.90],   # outliers
])
COL = {'core': '#2980b9', 'border': '#27ae60', 'noise': '#bdc3c7'}

def classify(P, eps, min_samples):
    """Return per-point class using the DBSCAN rule (a point counts itself)."""
    within = np.linalg.norm(P[:, None] - P[None, :], axis=2) <= eps
    counts = within.sum(1)
    is_core = counts >= min_samples
    cls = np.where(is_core, 'core', 'noise').astype(object)
    for i in range(len(P)):
        if not is_core[i] and np.any(within[i] & is_core):
            cls[i] = 'border'
    return within, counts, cls

# Scan the points left-to-right so the walk reads naturally
ORDER = np.lexsort((P[:, 1], P[:, 0]))
PAD = 0.6
XLIM = (P[:, 0].min() - PAD, P[:, 0].max() + PAD)
YLIM = (P[:, 1].min() - PAD, P[:, 1].max() + PAD)

def dbscan_walk(eps=0.45, min_samples=4, step=1):
    within, counts, cls = classify(P, eps, min_samples)
    n = len(P)
    step = min(step, n)
    cur = ORDER[step - 1]
    done = set(ORDER[:step - 1])

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    # points already visited -> faint final colour; not-yet-visited -> blank
    for i in range(n):
        if i == cur:
            continue
        if i in done:
            ax.scatter(*P[i], s=80, color=COL[cls[i]], alpha=0.55, zorder=2)
        else:
            ax.scatter(*P[i], s=80, color='#ecf0f1', edgecolors='#bbbbbb', zorder=2)

    # neighbours of the current point: their eps-circles, link lines, and rings
    nbrs = [j for j in np.where(within[cur])[0] if j != cur]
    for j in nbrs:
        ax.add_patch(plt.Circle(P[j], eps, fill=False, color='#95a5a6',
                                lw=1, ls=':', alpha=0.35, zorder=1))
        ax.plot([P[cur, 0], P[j, 0]], [P[cur, 1], P[j, 1]],
                color='#e74c3c', lw=0.9, alpha=0.5, zorder=1)
        ax.scatter(*P[j], s=110, facecolors='none', edgecolors='#e74c3c',
                   lw=1.6, zorder=4)

    # the current point + its own eps-circle
    ax.add_patch(plt.Circle(P[cur], eps, fill=False, color='#e74c3c',
                            lw=2, ls='--', zorder=3))
    ax.scatter(*P[cur], s=190, color=COL[cls[cur]], edgecolors='k', lw=1.6, zorder=5)

    c = counts[cur]
    rel = '>=' if c >= min_samples else '<'
    note = {'core': 'enough neighbours',
            'border': 'too few, but touches a core',
            'noise': 'too few, no core nearby'}[cls[cur]]
    ax.set_title(f'point {step}/{n}: {c} within eps ({rel} min_samples={min_samples})'
                 f'  ->  {cls[cur].upper()}\\n({note})', fontsize=11)
    ax.legend(handles=[Line2D([], [], marker='o', ls='', color=COL[k],
                              label=k, markersize=9) for k in COL],
              loc='upper left', fontsize=9, framealpha=0.9)
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    plt.show()

interact(dbscan_walk,
         eps=FloatSlider(min=0.2, max=1.2, step=0.05, value=0.45, readout_format='.2f'),
         min_samples=IntSlider(min=2, max=6, step=1, value=4),
         step=IntSlider(min=1, max=len(P), step=1, value=1));

### Think about it

- Drag **`step`** through all the points. Watch the title report the neighbour count
  against `min_samples` and assign **core / border / noise** for each one.
- Find a **border** point: too few neighbours to be core itself, but sitting inside a
  core point's circle. It joins the cluster on someone else's density, not its own.
- Now raise **`min_samples`** to 6. Points that were core become border or noise — the
  whole cluster can dissolve. Then lower **`eps`**: same effect. Both knobs control the
  *same thing* — how dense "dense enough" has to be.
- An outlier only becomes core if you make `eps` large enough to reach it. Try it —
  and notice that's exactly how too-large `eps` starts merging everything (Part 3).

---
## Part 3: Tuning eps and min_samples

`eps` is the most consequential knob. Too small and everything fragments into noise;
too large and distinct clusters merge into one blob.

In [ ]:
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
param_sets = [(0.1, 5, 'eps=0.1 (too small)'),
              (0.2, 5, 'eps=0.2 (good)'),
              (0.5, 5, 'eps=0.5 (too large)')]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), gridspec_kw={'wspace': 0.2})
for ax, (eps, ms, title) in zip(axes, param_sets):
    db = DBSCAN(eps=eps, min_samples=ms).fit(X_moons)
    n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise = int(np.sum(db.labels_ == -1))
    ax.scatter(X_moons[:, 0], X_moons[:, 1], s=15, c=dbcolors(db.labels_), alpha=0.7)
    ax.set_title(f'{title}\n{n_cl} clusters, {n_noise} noise', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### Think about it

- At `eps=0.1`, almost everything is grey. Why does too-small `eps` turn real cluster
  members into noise?
- At `eps=0.5`, the two moons merge. What's the geometric reason — what got pulled
  into a single neighbourhood chain?
- Eyeballing `eps` like this doesn't scale. Part 4 makes it systematic.

---
## Part 4: The k-distance plot

A principled way to choose `eps`: for each point, measure the distance to its
`min_samples`-th nearest neighbour, then sort those distances. Cluster points (dense)
sit on a low, flat **plateau**; noise points (sparse) form a steep **tail**. The
**knee** between them is a good `eps`.

In [ ]:
Xb, _ = make_blobs(n_samples=300, centers=[[0, 0], [4, 4], [0, 5]],
                   cluster_std=0.4, random_state=0)
bg = np.random.uniform(-2, 7, size=(60, 2))
X_kd = np.vstack([Xb, bg])
k = 5

### Exercise: compute and plot the k-distance curve

**Hints:**
- `nn = NearestNeighbors(n_neighbors=k).fit(X_kd)`
- `distances, _ = nn.kneighbors(X_kd)` returns an `(n, k)` array; column `k-1` is the
  distance to the k-th neighbour (column 0 is the point itself, distance 0).
- Sort that column ascending with `np.sort`, then plot it.

In [ ]:
# YOUR CODE HERE
# Compute k_dist = sorted distances to the k-th nearest neighbour, then plot.
raise NotImplementedError("Build the k-distance curve")

plt.plot(k_dist, 'b-', lw=2)
plt.axhline(0.5, color='#e74c3c', ls='--', label='knee ~ eps=0.5')
plt.xlabel('points sorted by k-distance')
plt.ylabel(f'distance to {k}-th NN')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Read eps off the knee and cluster with it
db = DBSCAN(eps=0.5, min_samples=k).fit(X_kd)
n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
plt.scatter(X_kd[:, 0], X_kd[:, 1], s=14, c=dbcolors(db.labels_), alpha=0.8)
plt.title(f'eps read off the knee -> {n_cl} clusters, noise in grey')
plt.xticks([]); plt.yticks([])
plt.show()

### Think about it

- The knee is sharp here *because* the clusters sit in a sparse background — exactly
  DBSCAN's assumption. With uniform density (no noise), the curve is smooth and `eps`
  is genuinely ambiguous. Try rebuilding `X_kd` without the `bg` noise and re-plot.
- Does the knee `eps` recover the three blobs cleanly?

---
## Part 5: Failure modes

### 5a. Variable density

One `eps` can't serve both a tight cluster and a diffuse one.

In [ ]:
dense = np.random.randn(200, 2) * 0.3 + [-1, 0]
sparse = np.random.randn(50, 2) * 1.5 + [3, 0]
X_mixed = np.vstack([dense, sparse])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={'wspace': 0.3})
for ax, eps, title in [(axes[0], 0.5, 'Small eps - sparse cluster -> noise'),
                       (axes[1], 1.5, 'Large eps - everything merges')]:
    db = DBSCAN(eps=eps, min_samples=5).fit(X_mixed)
    ax.scatter(X_mixed[:, 0], X_mixed[:, 1], s=15, c=dbcolors(db.labels_), alpha=0.7)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### 5b. Chaining

DBSCAN grows clusters by *density-reachability* — walking point to point through
`eps`-neighbourhoods. A thin **bridge** of points chains two distinct clusters into one.

In [ ]:
c0 = np.random.randn(120, 2) * 0.35 + [0, 0]
c1 = np.random.randn(120, 2) * 0.35 + [4, 0]
bridge = np.column_stack([np.linspace(0.6, 3.4, 14), np.random.randn(14) * 0.08])
eps, ms = 0.45, 5

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={'wspace': 0.3})
for ax, data, title in [(axes[0], np.vstack([c0, c1]), 'Clean gap'),
                        (axes[1], np.vstack([c0, c1, bridge]), 'Thin bridge')]:
    db = DBSCAN(eps=eps, min_samples=ms).fit(data)
    n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    ax.scatter(data[:, 0], data[:, 1], s=14, c=dbcolors(db.labels_), alpha=0.8)
    ax.set_title(f'{title} -> {n_cl} cluster(s)', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### Think about it

- In 5a, is there *any* single `eps` that captures both clusters correctly? (This is
  the motivation for variants like HDBSCAN.)
- In 5b, the same `eps` that *separates* the clusters *merges* them once a sparse
  bridge appears. The bridge points are individually unremarkable — why are they so
  destructive?

---
## Part 6: A pipeline — reduce, then cluster

In high dimensions distances **concentrate**: every pair of points ends up roughly
equally far apart (the same L1/L2 metrics from notebook 02 stop discriminating), and
density-based clustering breaks. A standard fix is **reduce first, then cluster** —
strip the noise dimensions with PCA, then run the clustering in the low-D space.

In [ ]:
from sklearn.decomposition import PCA

# 3 clusters living in 2D, embedded in 102D with 100 pure-noise dimensions
centers = np.array([[0, 0], [5, 0], [2.5, 4.3]])
core = np.vstack([np.random.randn(150, 2) * 0.6 + c for c in centers])
noise_dims = np.random.randn(450, 100) * 0.8
X_hd = np.hstack([core, noise_dims])

### Exercise: cluster raw 102-D vs. PCA-reduced 2-D

Run DBSCAN twice and compare:

1. On the raw 102-D data `X_hd`, with `eps=9, min_samples=5`.
2. On a 2-D PCA projection of `X_hd`, with `eps=1.0, min_samples=5`.

For each, count the clusters and the noise points.

**Hints:**
- `X_2d = PCA(n_components=2).fit_transform(X_hd)`
- Reuse the `n_clusters` / `n_noise` pattern from Part 1.

In [ ]:
# YOUR CODE HERE
# 1. db_raw on X_hd (eps=9). 2. X_2d = PCA->2D, db_pca on X_2d (eps=1.0).
# Compute (n_clusters, n_noise) for each.
raise NotImplementedError("Cluster raw vs PCA-reduced")

print(f'Raw 102-D : {n_raw} clusters, {noise_raw} noise')
print(f'PCA -> 2-D: {n_pca} clusters, {noise_pca} noise')

In [ ]:
# Visualise both, using the same 2-D PCA projection for display
X_2d = PCA(n_components=2).fit_transform(X_hd)
db_raw = DBSCAN(eps=9, min_samples=5).fit(X_hd)
db_pca = DBSCAN(eps=1.0, min_samples=5).fit(X_2d)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={'wspace': 0.3})
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], s=12, alpha=0.5, c=dbcolors(db_raw.labels_))
axes[0].set_title('DBSCAN on raw 102-D', fontsize=11)
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], s=12, alpha=0.5, c=dbcolors(db_pca.labels_))
axes[1].set_title('PCA -> 2-D -> DBSCAN', fontsize=11)
for ax in axes:
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### Think about it

- In 102-D, most points were labelled noise even though the data really is three
  clean clusters. What did the 100 noise dimensions do to the pairwise distances?
- After PCA the same data clusters cleanly. PCA didn't *create* the structure — it
  removed the dimensions that were burying it.
- This "reduce -> cluster -> validate" loop is exactly the pipeline you'll run on real
  pulsar data in notebook 10.